In [7]:
import pandas as pd
import requests
import json
import time
from bs4 import BeautifulSoup
from tqdm import tqdm
import re

In [8]:
# UFC stats uses pagination by letter (A-Z)
# We need to scrape each letter separately
base_url = "http://ufcstats.com"
# Generate URLs for each letter A-Z
alphas = [chr(i) for i in range(ord("a"), ord("a") + 26)]
fighter_group_urls = [
    f"http://ufcstats.com/statistics/fighters?char={alpha}&page=all"
    for alpha in alphas
]

print(f"Will scrape fighters from {len(fighter_group_urls)} letter pages (A-Z)")
print(f"Example URLs:")
print(fighter_group_urls[:3])
print("...")

Will scrape fighters from 26 letter pages (A-Z)
Example URLs:
['http://ufcstats.com/statistics/fighters?char=a&page=all', 'http://ufcstats.com/statistics/fighters?char=b&page=all', 'http://ufcstats.com/statistics/fighters?char=c&page=all']
...


In [9]:
# Use requests instead of Selenium for the main page
# The fighters page should be accessible via simple HTTP request
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
})

In [10]:
# Note: We'll fetch each letter page separately in the next cell
# No need to fetch a single main page anymore
pass

In [11]:
# Fetch and parse fighter links from ALL letter pages (A-Z)
# Note: Fighter names are split across 3 links (first name, last name, detail link)
# Every 3rd link is the actual fighter detail page
all_link_list = []

print("Fetching fighter links from all letter pages (A-Z)...")
for letter_url in tqdm(fighter_group_urls, desc="Processing letters"):
    try:
        response = session.get(letter_url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the table body with fighter links
        table = soup.find("tbody")
        if table:
            # Find all fighter links in this table
            # The pattern is: first name link, last name link, detail link (with href)
            links = table.findAll("a", {"class": "b-link b-link_style_black"}, href=True)
            
            # Every 3rd link (index 2, 5, 8, etc.) is the fighter detail link
            for i in range(2, len(links), 3):
                href = links[i].get('href')
                if href and href.startswith('http'):
                    all_link_list.append(href)
        
        # Small delay between requests
        time.sleep(0.3)
    except Exception as e:
        print(f"\nError fetching {letter_url}: {e}")
        continue

print(f"\nFound {len(all_link_list)} total fighter links from all pages")

Fetching fighter links from all letter pages (A-Z)...


Processing letters:   0%|                                                                                                                             | 0/26 [00:00<?, ?it/s]/var/folders/ks/3f3cxycs04d3zm70f0bn7fjh0000gn/T/ipykernel_14793/2551112210.py:18: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  links = table.findAll("a", {"class": "b-link b-link_style_black"}, href=True)
Processing letters: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 26/26 [00:26<00:00,  1.00s/it]


Found 4451 total fighter links from all pages


In [12]:
# Remove duplicates while preserving order
output = []
seen = set()
for x in all_link_list:
    if x not in seen:
        seen.add(x)
        output.append(x)

print(f"Found {len(output)} unique fighter links after removing duplicates")
print(f"First few links:")
for link in output[:5]:
    print(f"  {link}")
print(f"\nLast few links:")
for link in output[-5:]:
    print(f"  {link}")

Found 4451 unique fighter links after removing duplicates
First few links:
  http://ufcstats.com/fighter-details/93fe7332d16c6ad9
  http://ufcstats.com/fighter-details/15df64c02b6b0fde
  http://ufcstats.com/fighter-details/59a9d6dac61c2540
  http://ufcstats.com/fighter-details/4961467134abd8be
  http://ufcstats.com/fighter-details/b361180739bed4b0

Last few links:
  http://ufcstats.com/fighter-details/02d808afb96669de
  http://ufcstats.com/fighter-details/1291dd6b8ab4d952
  http://ufcstats.com/fighter-details/523af801b3429015
  http://ufcstats.com/fighter-details/392878af4680849f
  http://ufcstats.com/fighter-details/0c277f3ff66b0208


In [13]:
# Scrape fighter data from each link
fighters_data = []

for url in tqdm(output, desc="Scraping fighters"):
    try:
        react = session.get(url, timeout=10)
        react.raise_for_status()
        soup = BeautifulSoup(react.content, 'lxml')

        # Extract name
        name_elem = soup.select_one('.b-content__title-highlight')
        if not name_elem:
            continue
        name = name_elem.text.strip()
        
        # Extract record
        record_elem = soup.select_one('.b-content__title-record')
        if not record_elem:
            continue
        record = record_elem.text.strip().split()[-1]
        
        # Extract nickname - handle empty nicknames properly
        nick_elem = soup.select_one('.b-content__Nickname')
        if nick_elem:
            nick = nick_elem.get_text(strip=True)
            # If nickname element exists but is empty, set to empty string
            if not nick:
                nick = ""
        else:
            nick = ""

        fighter = {"Name": name, "Record": record, "Nickname": nick}

        # Extract stats - the stats are in list items
        # Structure: <li class="b-list__box-list-item b-list__box-list-item_type_block">
        #   <i class="b-list__box-item-title">Stat Name:</i>
        #   Value (as text node after the <i> tag)
        list_items = soup.select('li.b-list__box-list-item')
        
        for item in list_items:
            # Find the title element (stat name)
            title_elem = item.find('i', class_='b-list__box-item-title')
            if title_elem:
                stat = title_elem.get_text(strip=True)
                
                # Get the text node that comes after the <i> tag
                # This is the actual value
                value = ""
                next_node = title_elem.next_sibling
                
                # Navigate through any whitespace nodes to find the actual value
                while next_node:
                    if isinstance(next_node, str):
                        text = next_node.strip()
                        if text:
                            value = text
                            break
                    elif hasattr(next_node, 'get_text'):
                        # If it's an element, get its text
                        text = next_node.get_text(strip=True)
                        if text:
                            value = text
                            break
                    next_node = next_node.next_sibling
                
                # If we didn't find a value from next_sibling, try getting all text and removing the stat name
                if not value or value == "--":
                    full_text = item.get_text(strip=True)
                    # Remove the stat name (with or without colon)
                    if stat.endswith(':'):
                        stat_to_remove = stat
                    else:
                        stat_to_remove = stat + ':'
                    
                    if full_text.startswith(stat_to_remove):
                        value = full_text[len(stat_to_remove):].strip()
                    elif full_text.startswith(stat):
                        value = full_text[len(stat):].strip()
                
                # Only add if we have a valid value (not "--", not empty, not just the stat name)
                if value and value != "--" and value != stat and value.strip():
                    fighter[stat] = value.strip()
        
        fighters_data.append(fighter)
        
        # Be polite - add a small delay to avoid overwhelming the server
        time.sleep(0.5)
        
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        continue

print(f"\nScraped data for {len(fighters_data)} fighters")


Scraping fighters:  36%|███████████████████████████████████████▉                                                                      | 1618/4451 [22:03<21:47:46, 27.70s/it]

Error scraping http://ufcstats.com/fighter-details/d85cf12439945941: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))


Scraping fighters:  38%|█████████████████████████████████████████▉                                                                     | 1681/4451 [23:09<3:32:54,  4.61s/it]

Error scraping http://ufcstats.com/fighter-details/483b3170b3c05cf0: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))


Scraping fighters:  39%|███████████████████████████████████████████▏                                                                  | 1746/4451 [25:30<21:37:45, 28.79s/it]

Error scraping http://ufcstats.com/fighter-details/8b9197746bd3e005: HTTPConnectionPool(host='ufcstats.com', port=80): Read timed out. (read timeout=10)


... (6 outputs left out)

Scraping fighters:  97%|███████████████████████████████████████████████████████████████████████████████████████████████████████▊   | 4318/4451 [1:18:16<11:13:21, 303.77s/it]

Error scraping http://ufcstats.com/fighter-details/eb29176b03dfa888: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))


Scraping fighters: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4451/4451 [1:19:58<00:00,  1.08s/it]


Scraped data for 4444 fighters


In [14]:
# Convert to DataFrame and save
if fighters_data:
    df_fighters = pd.DataFrame(fighters_data)
    df_fighters.to_csv('ufc_fighters_raw.csv', index=False)
    print(f"Saved {len(df_fighters)} fighters to ufc_fighters_raw.csv")
    print(f"\nColumns: {df_fighters.columns.tolist()}")
    print(f"\nFirst few rows:")
    df_fighters.head()
else:
    print("No fighter data collected!")

Saved 4444 fighters to ufc_fighters_raw.csv

Columns: ['Name', 'Record', 'Nickname', 'Weight:', 'DOB:', 'SLpM:', 'Str. Acc.:', 'SApM:', 'Str. Def:', 'TD Avg.:', 'TD Acc.:', 'TD Def.:', 'Sub. Avg.:', 'Height:', 'STANCE:', 'Reach:']

First few rows:
